## KELT-20b Evolution and Parameter Sweep

### Known Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
np.random.seed(0)

# Stellar parameters - KELT-20
stellar_mass = 2.3  # solar masses
stellar_radius = 1.6 * 0.00465  # AU (1.6 solar radii)
k2_s = 0.03 # Love number

# Planetary parameters - KELT-20b
planetary_mass = 3.382 * 0.000954588  # solar masses (3.382 Jupiter masses)
planetary_radius = 0.1 * 0.00465
k2_p = 0.5

G = 4*np.pi**2  # AU^3 / yr^2 Msun

# Initial spin
omega_0 = 0

# Final conditions
a_final = 0.0542
e_final = 0
star_age = 1e8

### Calculated Parameters

In [ ]:
# Initial orbital geometry
a_array = np.linspace(2, 5, 20)
e_array = 1 - 0.025 / a_array

# Convective friction time
m_cgs = 2.3 * 2e33  # stellar mass in grams
r_cgs = 1.6 * 7e10  # stellar radius in cm
l_cgs = 15.7 * 4e33  # stellar luminosity in erg/s

t_f_s = (m_cgs * r_cgs **2 / l_cgs)**(1/3)  # seconds
t_f = t_f_s / 31556952  # years

# Tidal time lag
tau = 2 * stellar_radius**3 / (G * stellar_mass * t_f)
tau = tau * 7

# 10% error
tau_10 = 0.1*tau
tau_10

tau_lower = tau - tau_10
tau_upper = tau + tau_10

tau_array = np.linspace(tau_lower, tau_upper, 20)

print(a_array, tau_array)

### Hut Model

In [ ]:
def f1_hut(e2):
    return 1 + (31/2)*e2 + (255/8)*e2**2 + (185/16)*e2**3 + (25/64)*e2**4

def f2_hut(e2):
    return 1 + (15/2)*e2 + (45/8)*e2**2 + (5/16)*e2**3

def f3_hut(e2):
    return 1 + (15/4)*e2 + (15/8)*e2**2 + (5/64)*e2**3

def f4_hut(e2):
    return 1 + (3/2)*e2 + (1/8)*e2**2

def f5_hut(e2):
    return 1 + 3*e2 + (3/8)*e2**2


def hut_model(params, t, mstar, rstar, k2star, mplanet, tau):
    a, e, omega = params
    
    global done
    if e < 1e-3:
        if done == "no":
            t_circ_list.append(t)
            done = "yes"
    
    if a < rstar:
        return [0,0,0]
    
    k =  k2star
    
    n = np.sqrt(G*(mstar + mplanet)/a**3)

    T = rstar**3 / (G * mstar *  tau) 

    q = mplanet / mstar

    rg = 1
    
    da_dt = -6 * (k / T) * q * (1 + q) * (rstar / a)**8 * a * (1 - e**2)**(-15/2) * (
         f1_hut(e**2) - (1 - e**2)**(3/2) * f2_hut(e**2) * omega / n)

    de_dt = -27 * (k / T) * q * (1 + q) * (rstar / a)**8 * e * (1 - e**2)**(-13/2) * (
        f3_hut(e**2) - 11/18 * (1 - e**2)**(3/2) * f4_hut(e**2) * omega / n)

    domega_dt = 3 * (k / T) * (q / rg)**2 * (rstar / a)**6 * n * (1-e**2)**(-6) * (
        f2_hut(e**2) - (1 - e**2)**(3/2) * f5_hut(e**2) * omega / n)

    
    return [da_dt, de_dt, domega_dt]


### Preliminary Results

In [ ]:
t_circ_list = []
a_list = []
tau_list = []

# Time
tmax = 1e9
Nout = 2000
hut_times = np.linspace(0, tmax, Nout)

# Setting up plot
# fig, ax = plt.subplots(2, 1, figsize = (5,8))

for i in range(len(tau_array)):
    tau = tau_array[i]
    
    for i in range(len(a_array)):
        a0 = a_array[i]
        e0 = e_array[i]

        done = "no"
        
        a_list.append(a0)
        tau_list.append(tau)
        
        initial_cond = [a0, e0, omega_0] 
        
        hut_solution = odeint(hut_model, initial_cond, hut_times, 
                                   args = (stellar_mass, stellar_radius, k2_s, 
                                          planetary_mass, tau), mxords = 12)
        
        # a_sol = hut_solution[:, 0]
        # e_sol = hut_solution[:, 1]
        
        # a0 = round(a0, 1)
        # e0 = round(e0, 3)
        
#         ax[0].plot(hut_times, a_sol, label = f"{a0} AU")
#         ax[1].plot(hut_times, e_sol, label = f"{e0}")

# ax[0].set_title("Hut Tidal Evolution")
# ax[0].set_ylabel("Distance (AU)")
# ax[0].set_xscale("log")
# ax[0].axhline(y = a_final, color = "k", linestyle = "dashed", label = "target")
# ax[0].axvline(x = star_age, color = "k", linestyle = "dotted", label = "age")
# ax[0].axvline(x = time, color = "k", linestyle = "dashdot", label = "t_circ")

# ax[1].set_ylabel(r'$e$')
# ax[1].set_xlabel("Time (yr)")
# ax[1].set_xscale("log")
# ax[1].axhline(y = 0, color = "k", linestyle = "dashed", label = "target")
# ax[1].axvline(x = star_age, color = "k", linestyle = "dotted", label = "age")
# ax[1].axvline(x = time, color = "k", linestyle = "dashdot", label = "t_circ")


# ax[0].legend()
# ax[1].legend()



In [ ]:
plt.scatter(a_list, tau_list, c = t_circ_list)
plt.xlabel("Semi-Major Axis (AU)")
plt.ylabel("Time Lag (yr)")
plt.colorbar(label = "Circularization Time (yr)")

### Plotting Distance and e vs Time

In [ ]:
# Time
tmax = 1e9
Nout = 2000
hut_times = np.linspace(0, tmax, Nout)

# Setting up plot
fig, ax = plt.subplots(2, 1, figsize = (5,8))

for i in range(len(a_array)):
    a0 = a_array[i]
    e0 = e_array[i]
    
    initial_cond = [a0, e0, omega_0] 
    
    hut_solution = odeint(hut_model, initial_cond, hut_times, 
                               args = (stellar_mass, stellar_radius, k2_s, 
                                      planetary_mass, tau), mxords = 12)

    
    a_sol = hut_solution[:, 0]
    e_sol = hut_solution[:, 1]
    
    a0 = round(a0, 1)
    e0 = round(e0, 3)
    
    ax[0].plot(hut_times, a_sol, label = f"{a0} AU")
    ax[1].plot(hut_times, e_sol, label = f"{e0}")

ax[0].set_title("Hut Tidal Evolution")
ax[0].set_ylabel("Distance (AU)")
ax[0].set_xscale("log")
ax[0].axhline(y = a_final, color = "k", linestyle = "dashed", label = "target")
ax[0].axvline(x = star_age, color = "k", linestyle = "dotted", label = "age")

ax[1].set_ylabel(r'$e$')
ax[1].set_xlabel("Time (yr)")
ax[1].set_xscale("log")
ax[1].axhline(y = 0, color = "k", linestyle = "dashed", label = "target")
ax[1].axvline(x = star_age, color = "k", linestyle = "dotted", label = "age")

# ax[0].legend()
# ax[1].legend()